# Wine Cultivar Origin Prediction System
## Part A - Model Development

This notebook builds a machine learning model to predict wine cultivar based on chemical properties.

### Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

### Step 2: Load the Wine Dataset

In [ ]:
# Load wine dataset
wine_data = load_wine()

# Create DataFrame
df = pd.DataFrame(data=wine_data.data, columns=wine_data.feature_names)
df['cultivar'] = wine_data.target

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

### Step 3: Data Exploration

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Basic statistics
print("\nDataset Info:")
df.info()

print("\nBasic Statistics:")
df.describe()

In [ ]:
# Check class distribution
print("\nClass Distribution:")
print(df['cultivar'].value_counts())

# Visualize class distribution
plt.figure(figsize=(8, 5))
df['cultivar'].value_counts().plot(kind='bar', color=['#ff6b6b', '#4ecdc4', '#45b7d1'])
plt.title('Wine Cultivar Distribution')
plt.xlabel('Cultivar Class')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Step 4: Feature Selection

Selecting 6 features from the available features (excluding cultivar):
1. alcohol
2. malic_acid
3. total_phenols
4. flavanoids
5. color_intensity
6. proline

In [ ]:
# Selected features
selected_features = [
    'alcohol',
    'malic_acid', 
    'total_phenols',
    'flavanoids',
    'color_intensity',
    'proline'
]

# Prepare features and target
X = df[selected_features]
y = df['cultivar']

print(f"Selected Features: {selected_features}")
print(f"\nFeature Matrix Shape: {X.shape}")
print(f"Target Vector Shape: {y.shape}")

### Step 5: Data Preprocessing - Feature Scaling

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

# Feature Scaling (Mandatory due to varying feature ranges)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling completed using StandardScaler")

### Step 6: Model Training - Random Forest Classifier

In [ ]:
# Initialize Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1
)

# Train the model
model.fit(X_train_scaled, y_train)

print("Random Forest Classifier trained successfully!")

### Step 7: Model Evaluation

In [ ]:
# Make predictions
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Cultivar 0', 'Cultivar 1', 'Cultivar 2']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Cultivar 0', 'Cultivar 1', 'Cultivar 2'],
            yticklabels=['Cultivar 0', 'Cultivar 1', 'Cultivar 2'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': selected_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='#4ecdc4')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance in Random Forest Model')
plt.tight_layout()
plt.show()

### Step 8: Save the Model and Scaler

In [ ]:
# Save the trained model
joblib.dump(model, 'wine_cultivar_model.pkl')
print("Model saved as 'wine_cultivar_model.pkl'")

# Save the scaler
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as 'scaler.pkl'")

# Save feature names for reference
joblib.dump(selected_features, 'selected_features.pkl')
print("Feature names saved as 'selected_features.pkl'")

### Step 9: Test the Saved Model

In [ ]:
# Load the saved model
loaded_model = joblib.load('wine_cultivar_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

# Test with a sample
sample_input = X_test.iloc[0:1]
sample_scaled = loaded_scaler.transform(sample_input)
prediction = loaded_model.predict(sample_scaled)

print("\nTest Prediction:")
print(f"Input features: {sample_input.values[0]}")
print(f"Predicted Cultivar: {prediction[0]}")
print(f"Actual Cultivar: {y_test.iloc[0]}")

### Summary

**Model Details:**
- Algorithm: Random Forest Classifier
- Number of Features: 6
- Selected Features: alcohol, malic_acid, total_phenols, flavanoids, color_intensity, proline
- Feature Scaling: StandardScaler
- Model Persistence: Joblib
- Target Classes: 3 wine cultivars (0, 1, 2)

**Model Performance:**
- The model shows excellent performance on the test set
- All metrics (accuracy, precision, recall, F1-score) are reported
- The model is saved and ready for deployment